# Explore here

Analizar si el sentimiento expresado en las 500 reseñas de Harbor House Café coincide con la valoración media de 4,5 estrellas, e identificar discrepancias y limitaciones del modelo.

In [29]:
from pathlib import Path
import pandas as pd

# Funciona si el notebook se ejecuta desde src o desde la raíz.
root = Path.cwd()
if root.name == "src":
    root = root.parent

df = pd.read_csv(root / "data/raw/reviews.csv")

print("Filas y columnas:", df.shape)
display(df.head())

Filas y columnas: (500, 3)


,review_id,rating,review_text
0,1,5,Visited Harbor House Café last weekend. Great ...
1,2,5,Tried Harbor House Café after seeing it recomm...
2,3,5,Tried Harbor House Café after seeing it recomm...
3,4,5,Visited Harbor House Café last weekend. The st...
4,5,5,Grabbed a quick coffee at Harbor House Café th...


Revisamos tipos de datos, valores vacíos y duplicados para decidir si hace falta limpiar las reseñas antes del análisis.

In [30]:
df.info()

print("\nValores vacíos por columna:")
display(df.isna().sum())

print("Filas completamente duplicadas:", df.duplicated().sum())
print("IDs repetidos:", df["review_id"].duplicated().sum())
print("Textos repetidos:", df["review_text"].duplicated().sum())
print(
    "Reseñas vacías o con solo espacios:",
    df["review_text"].fillna("").str.strip().eq("").sum()
)

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   review_id    500 non-null    int64
 1   rating       500 non-null    int64
 2   review_text  500 non-null    str  
dtypes: int64(2), str(1)
memory usage: 11.8 KB

Valores vacíos por columna:


review_id      0
rating         0
review_text    0
dtype: int64

Filas completamente duplicadas: 0
IDs repetidos: 0
Textos repetidos: 7
Reseñas vacías o con solo espacios: 0


El dataset contiene 500 reseñas completas e identificadores únicos. Hay siete repeticiones de texto, que conservamos porque no son filas duplicadas y podrían corresponder a reseñas distintas. Revisamos ahora las puntuaciones y la longitud de los textos.

In [31]:
print("Distribución de puntuaciones:")
distribution = (
    df["rating"]
    .value_counts()
    .sort_index()
    .rename_axis("Estrellas")
    .to_frame("Reseñas")
)
distribution["Porcentaje"] = (
    distribution["Reseñas"] / len(df) * 100
).round(1)
display(distribution)

print(f"Valoración media del dataset: {df['rating'].mean():.2f} / 5")
print("Puntuaciones fuera de 1–5:", (~df["rating"].between(1, 5)).sum())

print("\nLongitud de las reseñas en palabras:")
display(df["review_text"].str.split().str.len().describe())

print("\nTextos repetidos:")
display(
    df.loc[
        df["review_text"].duplicated(keep=False),
        ["review_id", "rating", "review_text"]
    ].sort_values("review_text")
)

Distribución de puntuaciones:


,Reseñas,Porcentaje
Estrellas,,
1,12,2.4
2,18,3.6
3,38,7.6
4,72,14.4
5,360,72.0


Valoración media del dataset: 4.50 / 5
Puntuaciones fuera de 1–5: 0

Longitud de las reseñas en palabras:


count    500.000000
mean      29.104000
std        4.824078
min       14.000000
25%       26.000000
50%       29.000000
75%       32.000000
max       42.000000
Name: review_text, dtype: float64


Textos repetidos:


,review_id,rating,review_text
50,51,4,Grabbed a quick coffee at Harbor House Café th...
112,113,4,Grabbed a quick coffee at Harbor House Café th...
176,177,2,Grabbed a quick coffee at Harbor House Café th...
194,195,2,Grabbed a quick coffee at Harbor House Café th...
291,292,5,Grabbed a quick coffee at Harbor House Café th...
483,484,5,Grabbed a quick coffee at Harbor House Café th...
40,41,5,Grabbed a quick coffee at Harbor House Café th...
107,108,5,Grabbed a quick coffee at Harbor House Café th...
109,110,3,Regular customer at Harbor House Café here. Av...
450,451,3,Regular customer at Harbor House Café here. Av...


Conservamos las 500 reseñas: no hay valores vacíos, textos en blanco ni identificadores duplicados. Mantenemos los textos repetidos porque pertenecen a registros con IDs distintos. No eliminamos palabras, signos de puntuación ni negaciones, ya que pueden aportar información al análisis de sentimiento.

In [32]:
average = df["rating"].mean()
positive_pct = df["rating"].isin([4, 5]).mean() * 100
median_words = df["review_text"].str.split().str.len().median()

print(f"Valoración media: {average:.2f}/5")
print(f"Reseñas con 4–5 estrellas: {positive_pct:.1f}%")
print(f"Longitud mediana: {median_words:.0f} palabras")
print(f"Textos repetidos adicionales: {df['review_text'].duplicated().sum()}")

Valoración media: 4.50/5
Reseñas con 4–5 estrellas: 86.4%
Longitud mediana: 29 palabras
Textos repetidos adicionales: 7


### Insights de la exploración

- La valoración media es 4,50/5, coherente con el contexto del negocio.
- El 86,4 % de las reseñas tiene 4–5 estrellas. Compararemos este porcentaje con el sentimiento positivo predicho a partir del texto.
- La longitud mediana es de 29 palabras; revisaremos también la longitud en tokens para detectar posibles truncamientos.
- Conservamos las 500 reseñas, incluidas siete repeticiones de texto con identificadores distintos.

### Plan de acción y elección del modelo

Usaremos `nlptown/bert-base-multilingual-uncased-sentiment`, el modelo exigido, sin entrenarlo de nuevo. Cargaremos una única instancia con una revisión fija y procesaremos las reseñas por lotes.

Mapearemos sus predicciones: 1–2 estrellas → Negativo; 3 → Neutral; 4–5 → Positivo.

Compararemos la distribución y la media predichas con las puntuaciones originales. Después inspeccionaremos posibles falsos negativos y una muestra de 20 reseñas. El modelo fue ajustado con reseñas de productos, por lo que su rendimiento sobre servicios debe comprobarse.

In [33]:
import sys

print("Python:", sys.version)
print("Entorno del notebook:", sys.executable)

Python: 3.13.18 (main, Nov  5 2026, 09:50:38) [GCC 14.2.0]
Entorno del notebook: /usr/local/bin/python


In [34]:
import torch
import transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

PyTorch: 2.8.0+cpu
Transformers: 4.57.1


### Carga del modelo

Cargamos el tokenizador y el modelo preentrenado una sola vez.
Guardamos una revisión concreta de Hugging Face para poder reproducir
el análisis con los mismos archivos en futuras ejecuciones.

In [35]:
from pathlib import Path
from huggingface_hub import model_info
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_ID = "nlptown/bert-base-multilingual-uncased-sentiment"

# Guardar la revisión la primera vez y reutilizarla después.
revision_file = root / "model_revision.txt"

if revision_file.exists():
    MODEL_REVISION = revision_file.read_text().strip()
else:
    MODEL_REVISION = model_info(MODEL_ID).sha
    revision_file.write_text(MODEL_REVISION + "\n")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION
)

model.eval()

print("Modelo:", MODEL_ID)
print("Revisión fija:", MODEL_REVISION)
print("Etiquetas:", model.config.id2label)

Modelo: nlptown/bert-base-multilingual-uncased-sentiment
Revisión fija: 8f6f4e3a8f70be4b65d3a4a8762b6d781cda240d
Etiquetas: {0: '1 star', 1: '2 stars', 2: '3 stars', 3: '4 stars', 4: '5 stars'}


In [36]:
from pathlib import Path
import pandas as pd

root = Path.cwd()
if root.name == "src":
    root = root.parent

df = pd.read_csv(root / "data/raw/reviews.csv")

print("Filas y columnas:", df.shape)
display(df.head())

Filas y columnas: (500, 3)


,review_id,rating,review_text
0,1,5,Visited Harbor House Café last weekend. Great ...
1,2,5,Tried Harbor House Café after seeing it recomm...
2,3,5,Tried Harbor House Café after seeing it recomm...
3,4,5,Visited Harbor House Café last weekend. The st...
4,5,5,Grabbed a quick coffee at Harbor House Café th...


In [37]:
print("Modelo cargado:", "model" in globals())
print("Tokenizador cargado:", "tokenizer" in globals())

if "model" in globals():
    print("Etiquetas:", model.config.id2label)

Modelo cargado: True
Tokenizador cargado: True
Etiquetas: {0: '1 star', 1: '2 stars', 2: '3 stars', 3: '4 stars', 4: '5 stars'}


In [38]:
from huggingface_hub.utils import disable_progress_bars
from transformers.utils import logging

disable_progress_bars()
logging.disable_progress_bar()

print("Cargando el modelo…")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION
)
model.eval()

print("Modelo cargado correctamente")
print("Revisión fija:", MODEL_REVISION)
print("Etiquetas:", model.config.id2label)

Cargando el modelo…
Modelo cargado correctamente
Revisión fija: 8f6f4e3a8f70be4b65d3a4a8762b6d781cda240d
Etiquetas: {0: '1 star', 1: '2 stars', 2: '3 stars', 3: '4 stars', 4: '5 stars'}


### Inferencia sobre las 500 reseñas

Procesamos los textos por lotes con el modelo ya cargado.
Convertimos sus etiquetas a estrellas y bandas de sentimiento:
1–2 = Negativo, 3 = Neutral, 4–5 = Positivo.

Comprobamos la longitud en tokens y registramos si algún texto
supera el límite del modelo y necesita truncarse.

In [39]:
import re
import torch

# Read the star values from the model's actual labels.
label_to_stars = {}
for label_id, label in model.config.id2label.items():
    match = re.fullmatch(r"([1-5]) stars?", label)
    if not match:
        raise ValueError(f"Etiqueta inesperada: {label}")
    label_to_stars[int(label_id)] = int(match.group(1))

texts = df["review_text"].tolist()
max_length = model.config.max_position_embeddings

token_lengths = [
    len(tokenizer.encode(text, truncation=False))
    for text in texts
]

predicted_stars = []
confidence_scores = []
batch_size = 16
device = next(model.parameters()).device

with torch.inference_mode():
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        ).to(device)

        probabilities = model(**inputs).logits.softmax(dim=-1)
        scores, label_ids = probabilities.max(dim=-1)

        predicted_stars.extend(
            label_to_stars[label_id]
            for label_id in label_ids.cpu().tolist()
        )
        confidence_scores.extend(scores.cpu().tolist())

results = df.copy()
results["predicted_stars"] = predicted_stars
results["sentiment"] = results["predicted_stars"].map({
    1: "Negativo",
    2: "Negativo",
    3: "Neutral",
    4: "Positivo",
    5: "Positivo"
})
results["model_score"] = confidence_scores
results["token_count"] = token_lengths
results["was_truncated"] = results["token_count"] > max_length
results["model_id"] = MODEL_ID
results["model_revision"] = MODEL_REVISION

print("Reseñas procesadas:", len(results))
print("Textos truncados:", results["was_truncated"].sum())

display(
    results[
        ["review_id", "rating", "predicted_stars", "sentiment", "model_score"]
    ].head(10)
)

Reseñas procesadas: 500
Textos truncados: 0


,review_id,rating,predicted_stars,sentiment,model_score
0,1,5,5,Positivo,0.716958
1,2,5,5,Positivo,0.982220
2,3,5,5,Positivo,0.662323
3,4,5,5,Positivo,0.989547
4,5,5,5,Positivo,0.982912
5,6,5,5,Positivo,0.333903
6,7,5,5,Positivo,0.798027
7,8,5,5,Positivo,0.823738
8,9,3,3,Neutral,0.646989
9,10,5,5,Positivo,0.575618


### Sentimiento predicho frente a las puntuaciones originales

Comparamos las bandas de sentimiento del modelo con las bandas de
las puntuaciones humanas. Calculamos también la media de estrellas
predicha y su diferencia respecto a la valoración original de 4,50/5.

Las predicciones son estimaciones del modelo, no etiquetas verificadas.

In [40]:
bands = {
    1: "Negativo",
    2: "Negativo",
    3: "Neutral",
    4: "Positivo",
    5: "Positivo"
}
order = ["Negativo", "Neutral", "Positivo"]

results["rating_sentiment"] = results["rating"].map(bands)

comparison = pd.DataFrame({
    "Reseñas según rating": (
        results["rating_sentiment"].value_counts().reindex(order, fill_value=0)
    ),
    "Reseñas según modelo": (
        results["sentiment"].value_counts().reindex(order, fill_value=0)
    )
})

comparison["Rating (%)"] = (
    comparison["Reseñas según rating"] / len(results) * 100
).round(1)

comparison["Modelo (%)"] = (
    comparison["Reseñas según modelo"] / len(results) * 100
).round(1)

display(comparison)

print(f"Media original: {results['rating'].mean():.2f}/5")
print(f"Media predicha: {results['predicted_stars'].mean():.2f}/5")
print(f"Reseñas procesadas: {len(results)}")
print(f"Textos truncados: {results['was_truncated'].sum()}")

,Reseñas según rating,Reseñas según modelo,Rating (%),Modelo (%)
Negativo,30,47,6.0,9.4
Neutral,38,41,7.6,8.2
Positivo,432,412,86.4,82.4


Media original: 4.50/5
Media predicha: 4.40/5
Reseñas procesadas: 500
Textos truncados: 0


### Interpretación inicial

La media predicha es 4,40/5, frente a 4,50/5 en las puntuaciones
originales: una diferencia de −0,10 estrellas. Se procesaron las
500 reseñas sin truncamiento.

Las medias son próximas, pero esto no demuestra que las predicciones
individuales sean correctas. Revisaremos posibles falsos negativos:
textos clasificados como negativos pese a tener 4–5 estrellas.

In [41]:
possible_false_negatives = results.loc[
    results["rating"].isin([4, 5])
    & results["predicted_stars"].isin([1, 2])
].copy()

print("Posibles falsos negativos:", len(possible_false_negatives))

high_ratings = results["rating"].isin([4, 5]).sum()
print(
    "Porcentaje entre reseñas con 4–5 estrellas:",
    f"{len(possible_false_negatives) / high_ratings * 100:.1f}%"
)

# Print complete texts so we can assess their wording.
for _, row in possible_false_negatives.head(10).iterrows():
    print(
        f"\nID: {row['review_id']} | "
        f"Rating: {row['rating']} | "
        f"Predicción: {row['predicted_stars']}"
    )
    print(row["review_text"])

Posibles falsos negativos: 16
Porcentaje entre reseñas con 4–5 estrellas: 3.7%

ID: 14 | Rating: 4 | Predicción: 1
Tried Harbor House Café after seeing it recommended online. Every dish was bursting with flavor. The team clearly cares about the customers. Already planning my next visit.

ID: 15 | Rating: 5 | Predicción: 1
Every dish was bursting with flavor. The place has such a cozy, welcoming vibe. Already planning my next visit.

ID: 73 | Rating: 5 | Predicción: 1
Stopped by Harbor House Café for the first time. Service was fast and friendly. Every dish was bursting with flavor. Worth every penny. Already planning my next visit.

ID: 87 | Rating: 5 | Predicción: 1
Tried Harbor House Café after seeing it recommended online. The seasonal menu never disappoints. They went out of their way to accommodate my allergy. Will definitely be back!

ID: 138 | Rating: 5 | Predicción: 2
Tried Harbor House Café after seeing it recommended online. They went out of their way to accommodate my allerg

### Revisión de posibles falsos negativos

Detectamos 16 reseñas con puntuación humana de 4–5 estrellas y
predicción de 1–2 estrellas: el 3,7 % de las reseñas con valoración alta.

Los ejemplos 14, 15, 73, 87, 138 y 200 contienen lenguaje claramente
positivo. Los ejemplos 204 y 322 combinan críticas al personal con
una valoración global favorable y requieren una interpretación más matizada.

Observamos expresiones repetidas, elogios al servicio y formulaciones
como “never disappoints”. Son patrones para investigar, no causas
demostradas del error. El desajuste entre reseñas de productos y
servicios podría contribuir a estas discrepancias.

In [42]:
for start in range(0, len(possible_false_negatives), 4):
    block = possible_false_negatives.iloc[start:start + 4]

    text = "\n\n".join(
        f"ID: {row['review_id']} | Rating: {row['rating']} | "
        f"Predicción: {row['predicted_stars']}\n{row['review_text']}"
        for _, row in block.iterrows()
    )

    display(pd.DataFrame({"Reseñas para revisar": [text]}).style.set_properties(
        **{"white-space": "pre-wrap", "text-align": "left"}
    ))

,Reseñas para revisar
0,"ID: 14 | Rating: 4 | Predicción: 1 Tried Harbor House Café after seeing it recommended online. Every dish was bursting with flavor. The team clearly cares about the customers. Already planning my next visit. ID: 15 | Rating: 5 | Predicción: 1 Every dish was bursting with flavor. The place has such a cozy, welcoming vibe. Already planning my next visit. ID: 73 | Rating: 5 | Predicción: 1 Stopped by Harbor House Café for the first time. Service was fast and friendly. Every dish was bursting with flavor. Worth every penny. Already planning my next visit. ID: 87 | Rating: 5 | Predicción: 1 Tried Harbor House Café after seeing it recommended online. The seasonal menu never disappoints. They went out of their way to accommodate my allergy. Will definitely be back!"


,Reseñas para revisar
0,"ID: 138 | Rating: 5 | Predicción: 2 Tried Harbor House Café after seeing it recommended online. They went out of their way to accommodate my allergy. The place has such a cozy, welcoming vibe. Already planning my next visit. ID: 200 | Rating: 5 | Predicción: 2 Stopped by Harbor House Café for the first time. The music and decor set the perfect mood. Every dish was bursting with flavor. Already planning my next visit. ID: 204 | Rating: 5 | Predicción: 1 Came to Harbor House Café for a birthday brunch. Every dish was bursting with flavor. The cashier was rude when i asked about the menu, but honestly the food made up for it. Already planning my next visit. ID: 254 | Rating: 5 | Predicción: 1 Stopped by Harbor House Café for the first time. They went out of their way to accommodate my allergy. Great spot to work or catch up with friends. Will definitely be back!"


,Reseñas para revisar
0,"ID: 270 | Rating: 5 | Predicción: 1 Stopped by Harbor House Café for the first time. Our server remembered our names by the second visit. Every dish was bursting with flavor. Great value for what you get. Will definitely be back! ID: 322 | Rating: 5 | Predicción: 1 Every dish was bursting with flavor. The staff seemed annoyed to be there, but honestly the food made up for it. Will definitely be back! ID: 332 | Rating: 5 | Predicción: 1 Stopped by Harbor House Café for the first time. They went out of their way to accommodate my allergy. The coffee was rich and perfectly brewed. Already planning my next visit. ID: 409 | Rating: 5 | Predicción: 1 Tried Harbor House Café after seeing it recommended online. Every dish was bursting with flavor. Service was fast and friendly. Already planning my next visit."


,Reseñas para revisar
0,ID: 440 | Rating: 5 | Predicción: 2 Regular customer at Harbor House Café here. Every dish was bursting with flavor. Our server remembered our names by the second visit. Reasonably priced for the quality. Already planning my next visit. ID: 462 | Rating: 5 | Predicción: 2 Tried Harbor House Café after seeing it recommended online. The team clearly cares about the customers. Every dish was bursting with flavor. Already planning my next visit. ID: 474 | Rating: 5 | Predicción: 1 Stopped by Harbor House Café for the first time. Every dish was bursting with flavor. Service was fast and friendly. This is now my go-to spot. ID: 489 | Rating: 5 | Predicción: 1 Tried Harbor House Café after seeing it recommended online. Every dish was bursting with flavor. The team clearly cares about the customers. Worth every penny. Already planning my next visit.


### Validación manual de 20 reseñas

Revisamos una muestra aleatoria de 15 reseñas y añadimos cinco casos
con discrepancias entre rating alto y predicción negativa.

Primero etiquetamos el texto sin consultar las predicciones.
Esta muestra sirve para explorar errores; al incluir casos problemáticos
deliberadamente, no representa una estimación de precisión global.

In [43]:
review_path = root / "data/processed/manual_review.csv"

# Recreate the same reproducible selection.
random_sample = results.sample(n=15, random_state=42).copy()
random_sample["selection_reason"] = "Aleatoria"

remaining = possible_false_negatives.loc[
    ~possible_false_negatives["review_id"].isin(random_sample["review_id"])
]

targeted_sample = remaining.sample(
    n=min(5, len(remaining)),
    random_state=42
).copy()
targeted_sample["selection_reason"] = "Discrepancia"

manual_sample = pd.concat(
    [random_sample, targeted_sample],
    ignore_index=True
)

# Preserve the annotations you already completed.
if not review_path.exists():
    raise FileNotFoundError(
        "The completed manual_review.csv is missing."
    )

manual_review = pd.read_csv(review_path, keep_default_na=False)

assert len(manual_review) == 20
assert manual_review["review_id"].is_unique
assert set(manual_review["review_id"]) == set(manual_sample["review_id"])

print("Loaded 20 saved manual reviews. Nothing overwritten.")

Loaded 20 saved manual reviews. Nothing overwritten.


In [44]:
valid_labels = {"Positivo", "Neutral", "Negativo"}

manual_review["manual_sentiment"] = (
    manual_review["manual_sentiment"].str.strip()
)

assert manual_review["manual_sentiment"].isin(valid_labels).all(), (
    "Some labels are missing or invalid."
)

assert manual_review["manual_note"].str.strip().ne("").all(), (
    "Some review notes are missing."
)

print("All 20 manual annotations are complete.")

All 20 manual annotations are complete.


### Comparación con la revisión manual

Comparamos las 20 etiquetas manuales con las predicciones del modelo.
Mostramos los desacuerdos y sus motivos.

La muestra incluye casos problemáticos seleccionados deliberadamente,
por lo que su concordancia no estima la precisión sobre las 500 reseñas.

In [45]:
manual_review = pd.read_csv(review_path, keep_default_na=False)

# Check that every review has a valid annotation.
valid_labels = {"Positivo", "Neutral", "Negativo"}
manual_review["manual_sentiment"] = (
    manual_review["manual_sentiment"].str.strip()
)

assert len(manual_review) == 20, "Expected 20 reviews."
assert manual_review["review_id"].is_unique, "There are repeated IDs."
assert manual_review["manual_sentiment"].isin(valid_labels).all(), (
    "Some sentiment labels are missing or invalid."
)
assert manual_review["manual_note"].str.strip().ne("").all(), (
    "Some reasons are missing."
)

manual_comparison = manual_review.merge(
    results[["review_id", "rating", "predicted_stars", "sentiment"]],
    on="review_id",
    how="left",
    validate="one_to_one"
)

assert manual_comparison["predicted_stars"].notna().all()

manual_comparison["agrees"] = (
    manual_comparison["manual_sentiment"]
    == manual_comparison["sentiment"]
)

print("Reviews checked:", len(manual_comparison))
print("Agreements:", manual_comparison["agrees"].sum())
print("Disagreements:", (~manual_comparison["agrees"]).sum())
print(
    "Agreement in this sample:",
    f"{manual_comparison['agrees'].mean() * 100:.1f}%"
)

print("\nManual labels versus model labels:")
display(
    pd.crosstab(
        manual_comparison["manual_sentiment"],
        manual_comparison["sentiment"],
        rownames=["Manual"],
        colnames=["Modelo"]
    ).reindex(
        index=["Negativo", "Neutral", "Positivo"],
        columns=["Negativo", "Neutral", "Positivo"],
        fill_value=0
    )
)

print("\nCases to discuss:")
display(
    manual_comparison.loc[
        ~manual_comparison["agrees"],
        [
            "review_id", "review_text", "manual_sentiment",
            "sentiment", "predicted_stars", "manual_note"
        ]
    ]
)

Reviews checked: 20
Agreements: 12
Disagreements: 8
Agreement in this sample: 60.0%

Manual labels versus model labels:


Modelo,Negativo,Neutral,Positivo
Manual,,,
Negativo,1,0,0
Neutral,0,3,3
Positivo,5,0,8



Cases to discuss:


,review_id,review_text,manual_sentiment,sentiment,predicted_stars,manual_note
3,156,Grabbed a quick coffee at Harbor House Café th...,Neutral,Positivo,4,Complains about the atmosphere
5,395,The food was absolutely delicious. We waited 2...,Neutral,Positivo,4,praises the food but not the service
10,10,The pastries were incredible. The staff seemed...,Neutral,Positivo,5,Praises the place and the food but not the ser...
15,14,Tried Harbor House Café after seeing it recomm...,Positivo,Negativo,1,"Praises the atmosphere, food and place and sug..."
16,15,Every dish was bursting with flavor. The place...,Positivo,Negativo,1,"Praises the atmosphere, food and place and sug..."
17,200,Stopped by Harbor House Café for the first tim...,Positivo,Negativo,2,Praises the atmosphere and place and suggest t...
18,474,Stopped by Harbor House Café for the first tim...,Positivo,Negativo,1,Praises the atmosphere and place and suggest t...
19,462,Tried Harbor House Café after seeing it recomm...,Positivo,Negativo,2,Praises the atmosphere and place and suggest t...


### Resultados de la revisión manual

De las 20 reseñas revisadas, 12 coinciden con el modelo y 8 discrepan:
una concordancia del 60 % en esta muestra.

En cinco casos, el modelo predice sentimiento negativo para textos
que la revisión manual considera positivos. En otros tres, predice
positivo para textos considerados neutrales por combinar elogios
con críticas al servicio.

La muestra incluye cinco posibles errores seleccionados expresamente.
Por ello, el 60 % no es una estimación de precisión global.
Las etiquetas manuales también implican criterio interpretativo,
especialmente en reseñas mixtas.

In [46]:
# Restore how each review was selected.
selection = manual_sample[["review_id", "selection_reason"]]

manual_comparison = manual_comparison.drop(
    columns=["selection_reason"], errors="ignore"
).merge(
    selection,
    on="review_id",
    how="left",
    validate="one_to_one"
)

print("Agreement by selection method:")
display(
    manual_comparison.groupby("selection_reason").agg(
        reviews=("agrees", "size"),
        agreements=("agrees", "sum"),
        agreement_pct=("agrees", lambda values: values.mean() * 100)
    ).round(1)
)

print("\nSentiment distribution across all 500 reviews:")
display(comparison[["Rating (%)", "Modelo (%)"]])

Agreement by selection method:


,reviews,agreements,agreement_pct
selection_reason,,,
Aleatoria,15,12,80.0
Discrepancia,5,0,0.0



Sentiment distribution across all 500 reviews:


,Rating (%),Modelo (%)
Negativo,6.0,9.4
Neutral,7.6,8.2
Positivo,86.4,82.4


### Conclusiones para la account manager

El sentimiento escrito respalda, en términos generales, la valoración
alta del negocio. El modelo clasifica el 82,4 % de las 500 reseñas como
positivas, el 8,2 % como neutrales y el 9,4 % como negativas.

Las puntuaciones originales contienen un 86,4 % de valoraciones de
4–5 estrellas. La proporción positiva predicha es cuatro puntos
porcentuales inferior. La media predicha, 4,40/5, también es ligeramente
inferior a la original de 4,50/5. Son medidas relacionadas, pero no
equivalentes: una puntuación y el sentimiento del texto pueden diferir.

Detectamos 16 posibles falsos negativos entre las reseñas con
4–5 estrellas. La revisión manual confirmó cinco casos seleccionados
en los que el texto se consideró positivo y el modelo predijo negativo.
También hubo discrepancias al interpretar reseñas con elogios y críticas.

La concordancia fue del 80 % en las 15 reseñas seleccionadas al azar.
La muestra es pequeña y procede de una sola persona, por lo que no
permite afirmar una precisión global del 80 %. El 60 % de concordancia
en las 20 reseñas tampoco es representativo, porque cinco se eligieron
deliberadamente por sus discrepancias.

Recomendamos utilizar el modelo para resumir tendencias y priorizar
revisiones, no para tratar cada etiqueta como una verdad confirmada.
Antes de comunicar problemas concretos al cliente, conviene revisar
los textos negativos y mixtos. El desajuste entre productos y servicios
es una posible limitación, cuya contribución no hemos demostrado.

In [47]:
processed_dir = root / "data/processed"
processed_dir.mkdir(parents=True, exist_ok=True)

results.to_csv(
    processed_dir / "reviews_with_sentiment.csv",
    index=False
)

manual_comparison.to_csv(
    processed_dir / "manual_review_comparison.csv",
    index=False
)

print("Enriched reviews saved:", len(results))
print("Manual comparisons saved:", len(manual_comparison))

Enriched reviews saved: 500
Manual comparisons saved: 20
